# IndicOCR — inference

Page image in, reading-ordered Markdown and per-block JSON out.

Two stages with a plain JSON file between them:

| stage | model | what it does |
| --- | --- | --- |
| 1 | **IndicDocLayout** (33M, PP-DocLayoutV3) | finds blocks, classifies them, orders them |
| 2 | **IndicBlockOCR** (0.8B, Qwen3.5, vLLM) | transcribes the textual blocks |

Either stage runs alone, and the intermediate layout can be inspected or hand-corrected
before stage 2 sees it — which is the main reason to work in a notebook rather than
running `scripts/ocr/parse.sh`.

**Needs a GPU.** Construct `IndicOCR` before anything else touches CUDA: it starts
the recognizer first so vLLM can initialise the device.

In [ ]:
from importlib.metadata import PackageNotFoundError, version

import torch

import bodhan_genai.ocr

print("bodhan-genai:", bodhan_genai.ocr.__version__)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
for package in ("transformers", "vllm"):
    try:
        print(f"{package}:", version(package))
    except PackageNotFoundError:
        print(f"{package}: not installed")

## Checkpoints

Leave these as `None` to pull the public `bodhan-ai/indic-ocr` from the Hub (no credentials
while it is private), or point them at local directories.

In [ ]:
import os

LAYOUT_CKPT = os.environ.get("BODHAN_OCR_LAYOUT_CKPT")  # None -> Hub
RECOGNIZER_CKPT = os.environ.get("BODHAN_OCR_RECOGNIZER_CKPT")
PAGE = "page.png"  # <- a page image to work on

print("layout:    ", LAYOUT_CKPT or "Hub")
print("recognizer:", RECOGNIZER_CKPT or "Hub")

## The contract

Before running anything, look at what the pipeline promises: the three prompts, which
block types are kept, which are never cropped, and the output schema.

Worth knowing: a label that is not in `LABEL_TO_TYPE` falls through to `Text` **by
design**. A mis-cased or typo'd label does not raise — it quietly gets the prose prompt
instead of the equation or table one.

In [ ]:
from bodhan_genai.ocr import KEPT_BLOCK_TYPES, OCR_SKIP_LABELS, TableFormat, prompt_for

print("kept block types:", sorted(KEPT_BLOCK_TYPES))
print()
print("never transcribed:", sorted(OCR_SKIP_LABELS))
print()
for block_type in ("Text", "Equation", "Table"):
    print(f"{block_type:10} -> {prompt_for(block_type, TableFormat.HTML)[:90]}...")

## Stage 1 — layout only

No recognizer is loaded here, so this is fast and needs no vLLM. Blocks come back
already ordered.

In [ ]:
from bodhan_genai.ocr import IndicDocLayout, LayoutConfig

layout = IndicDocLayout(LAYOUT_CKPT, config=LayoutConfig(conf=0.5))
blocks = layout.detect(PAGE)
layout.close()

print(f"{len(blocks)} blocks")
for block in blocks[:10]:
    x0, y0, x1, y1 = block.bbox_xyxy
    print(
        f"  {block.order:>3}  {block.label:<20} {block.type:<12} conf={block.conf:.2f}"
        f"  [{x0:.0f},{y0:.0f},{x1:.0f},{y1:.0f}]"
    )

### Look at it

Reading order is the thing most worth eyeballing — a detection mistake is obvious in the
numbers, an ordering mistake usually is not.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

image = Image.open(PAGE).convert("RGB")
figure, axis = plt.subplots(figsize=(9, 12))
axis.imshow(image)
for block in blocks:
    x0, y0, x1, y1 = block.bbox_xyxy
    axis.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, linewidth=1.2))
    axis.text(
        x0,
        y0 - 4,
        f"{block.order}:{block.label}",
        fontsize=7,
        bbox={"facecolor": "white", "alpha": 0.7, "pad": 0.5},
    )
axis.axis("off")
plt.tight_layout()

## Correcting a layout by hand

`JsonLayoutBackend` replays any layout of the documented schema — from stage 1, from
another detector, or hand-edited. It needs no torch, so a corrected layout can be
prepared anywhere and transcribed later.

In [ ]:
from bodhan_genai.ocr import JsonLayoutBackend, PageResult

page = PageResult(image=PAGE, width=image.width, height=image.height, blocks=list(blocks))

# e.g. drop a block the detector invented, then renumber by replaying it
page.blocks = [b for b in page.blocks if b.conf > 0.6]
replayed = JsonLayoutBackend(page).detect(PAGE)
print(
    f"{len(blocks)} blocks -> {len(replayed)} after filtering, ranks renumbered:",
    [b.order for b in replayed[:10]],
)

## Stage 2 — transcribe

This loads vLLM and takes a couple of minutes to start. Every block of the page goes
through in one continuous batch, so a whole directory costs barely more than one page —
pass a folder rather than a single image whenever you can.

In [ ]:
from bodhan_genai.ocr import IndicBlockOCR, RecognizerConfig

recognizer = IndicBlockOCR(
    RECOGNIZER_CKPT,
    config=RecognizerConfig(gpu_memory_utilization=0.85, max_tokens=1024),
)
result = recognizer.run(PAGE, page)
recognizer.close()

print(result.markdown[:1500])

## Both stages in one process

`IndicOCR` is the same thing end to end, and is what `scripts/ocr/parse.sh` runs.

In [ ]:
from bodhan_genai.ocr import IndicOCR

with IndicOCR(layout_ckpt=LAYOUT_CKPT, recognizer_ckpt=RECOGNIZER_CKPT) as parser:
    parsed = parser.parse(PAGE)

transcribed = sum(1 for b in parsed.blocks if (b.text or "").strip())
print(f"{len(parsed.blocks)} blocks, {transcribed} transcribed")
print(parsed.markdown[:1000])

### Tables: HTML by default

`colspan`, `rowspan` and in-cell line breaks have no GitHub-flavored-Markdown spelling,
so a merged-cell table rendered as Markdown silently loses its structure. Markdown is
available for flat tables — and it is what the published olmOCR number was measured
with.

In [ ]:
from bodhan_genai.ocr import TableFormat

with IndicOCR(
    layout_ckpt=LAYOUT_CKPT,
    recognizer_ckpt=RECOGNIZER_CKPT,
    recognizer_config=RecognizerConfig(table_format=TableFormat.MARKDOWN),
) as parser:
    flat = parser.parse(PAGE)

print(flat.markdown[:800])

## Over HTTP

For pages arriving over time, or callers without a GPU, run the recognizer behind stock
`vllm serve` (`scripts/ocr/serve.sh`) and keep layout client-side — it is a 133 MB
detector that runs about a second per page on CPU.

The endpoint serves the recognizer alone and expects one block crop per request, so use
`OCRClient` rather than pointing a bare OpenAI client at it.

In [ ]:
from bodhan_genai.ocr.serving import OCRClient

# with `scripts/ocr/serve.sh` running elsewhere:
# with OCRClient("http://localhost:8000/v1") as client:
#     print(client.parse(PAGE).markdown)
print(OCRClient.__doc__)

## Next

* `docs/ocr/end-to-end.md` — every workflow on one page
* `docs/ocr/contract.md` — prompts, taxonomy, output schema
* `notebooks/ocr/training.ipynb` — training the layout model